# اجرای پژوهشی پایان‌نامه در Google Colab

ابتدا README_FA و CONFORMANCE_FA را بخوانید. آموزش و آزمون با دادهٔ واقعی انجام می‌شود؛ خروجی ساختگی نداریم. هفت پارامتر معادلهٔ اصلی نامشخص و دستگاه پیوست متفاوت و مستعد واگرایی است. هیچ عدد فصل چهارم از پیش تأیید نشده است. این بسته برای دادهٔ شناسایی‌پذیر بیمار یا استفادهٔ بالینی نیست.

## ۱. بارگذاری ZIP بسته
فایل `medical_thesis_colab.zip` را انتخاب کنید؛ این فایل فقط شامل کد و مستندات است. نوت‌بوک قدیمی اجرا نمی‌شود.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess
from google.colab import files
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith('.zip')]
if len(archives) != 1: raise ValueError('دقیقاً ZIP بسته را انتخاب کنید')
ROOT = Path('/content/thesis_toolkit')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    for name in z.namelist():
        if not (ROOT/name).resolve().is_relative_to(ROOT.resolve()): raise ValueError('Unsafe zip')
    z.extractall(ROOT)
PACKAGE = ROOT/'medsec_colab'
os.chdir(PACKAGE)
sys.path.insert(0, str(PACKAGE))
print('مسیر بسته:', PACKAGE)

## ۲. نصب و آزمون نرم‌افزار
آزمون‌های خودکار از دادهٔ کوچک مصنوعی **فقط برای بررسی نرم‌افزار** استفاده می‌کنند؛ آن‌ها آزمایش پزشکی یا بازتولید نتایج نیستند.

In [ ]:
subprocess.run([sys.executable, 'scripts/install_colab.py'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'scripts/tests', '-q'], check=True)
from scripts.artifacts import environment
print(json.dumps(environment(), indent=2, ensure_ascii=False))

## ۳. مسیرهای ماندگار و تنظیمات
نام/نسخهٔ دقیق داده و مسیر واقعی آن را وارد کنید. مسیر WORK برای وزن و گزارش است؛ DATA_ROOT باید پوشهٔ داده‌های دریافت‌شده با مجوز باشد. برای هر دیتاست جدا اجرا کنید.

In [ ]:
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.environ['THESIS_WORKDIR'] = '/content/drive/MyDrive/Thesis_Research' if MOUNT_DRIVE else '/content/Thesis_Research'
WORK = Path(os.environ['THESIS_WORKDIR']); WORK.mkdir(parents=True, exist_ok=True)
DATASET = 'DRIVE'  # DRIVE / RITE / BraTS2020 / COVID19_CXR
DATA_ROOT = WORK/'data_sources'/DATASET
SOURCE = ''  # شناسه/نسخه و منشأ واقعی داده؛ خالی مجاز نیست
CXR_CSV = WORK/'cxr_index.csv'
MANIFEST = WORK/f'{DATASET}.jsonl'
WEIGHTS = WORK/'weights'/DATASET
PAYLOAD = WORK/'metadata.bin'  # فایل آزمایشی فاقد اطلاعات هویتی بیمار
RUN_DIR = WORK/'runs'/f'{DATASET}_001'  # در اجرای جدید نام تازه انتخاب کنید
PROFILE = PACKAGE/'scripts/thesis_reference.json'  # نسخه مستند تکمیل‌شده خودتان را جایگزین کنید
RUN_TRAINING = False
RESUME_TRAINING = False
RUN_EXPERIMENTS = False
RUN_NIST = False
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## ۴. ساخت فهرست دادهٔ واقعی
اگر ساختار شما متفاوت است، مطابق README فایل JSONL صریح تهیه کنید. این سلول هیچ داده‌ای دانلود یا جعل نمی‌کند. برای BraTS اسلایس‌ها بعد از تقسیم بیمار ساخته می‌شوند؛ DRIVE/RITE از گروه‌های مشترک استفاده می‌کنند.

In [ ]:
from scripts.prepare import prepare
from scripts.data import load_manifest, load_sample
if not MANIFEST.exists():
    prepare(DATASET, DATA_ROOT, MANIFEST, SOURCE, csv_path=CXR_CSV if DATASET == 'COVID19_CXR' else None,
            modalities=['flair', 't1', 't1ce', 't2'], stride=1)
records = load_manifest(MANIFEST)
from collections import Counter
print(Counter((r['dataset'], r['split']) for r in records))
example, reference_mask, prep = load_sample(records[0])
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(example, cmap='gray'); axes[0].set_title('Real processed input')
axes[1].imshow(reference_mask, cmap='gray'); axes[1].set_title('Reference ROI')
plt.show(); print(prep)

## ۵. آموزش و ارزیابی U-Net
آموزش به عددهای ODE نیاز ندارد. `RUN_TRAINING=True` را پس از بررسی داده فعال کنید؛ دوره‌ها و دیگر تنظیمات در کپی JSON قابل تغییرند. بهترین وزن فقط از اعتبارسنجی انتخاب می‌شود. با قطع نشست و همان تنظیمات، RESUME_TRAINING را فعال کنید.

In [ ]:
from scripts.training import train
from scripts.experiments import evaluate_segmentation
cfg = json.loads(PROFILE.read_text())
if RUN_TRAINING:
    print(train(MANIFEST, DATASET, cfg, WEIGHTS, DEVICE, resume=RESUME_TRAINING))
CHECKPOINT = WEIGHTS/'best.pt'
if CHECKPOINT.exists():
    evaluation = evaluate_segmentation(MANIFEST, DATASET, CHECKPOINT, WORK/'evaluation'/f'{DATASET}.json', DEVICE)
    print({k:v for k,v in evaluation.items() if k not in ('records','environment')})
else:
    print('وزن آموزش‌دیده موجود نیست؛ ارزیابی یا استنتاج با وزن تصادفی انجام نشد.')

## ۶. اعتبار روش پیش از رمزنگاری
تنظیمات معادلهٔ ۳–۲ عمداً ناقص‌اند. پروفایل پیوست جایگزین بی‌نام نیست. برای استفادهٔ آزمایشی از آن باید PROFILE را صریحاً تغییر دهید و نتیجه را تفسیر پیوست بنامید. عددهای مطلوب دلیل انتخاب پارامتر نیستند.

In [ ]:
from scripts.config import validate
from scripts.model import Predictor
from scripts.chaos import roi_digest, stream
from scripts.artifacts import write_json
METHOD_READY = False
try:
    validate(cfg)
    predictor = Predictor(CHECKPOINT, DEVICE)
    selected = next(r for r in records if r['split'] == 'test')
    image, target, preprocessing = load_sample(selected)
    predicted_roi = predictor(image)
    digest, moments = roi_digest(image, predicted_roi)
    raw = stream(digest, image.size, cfg, raw=True)
    METHOD_READY = True
    print('پیش‌بررسی این نمونه موفق؛ هنوز اثبات پایداری یا امنیت نیست.', moments)
except Exception as exc:
    write_json(WORK/'method_preflight.json', {'status':'blocked','error':f'{type(exc).__name__}: {exc}','config':cfg})
    print('مانع ثبت‌شده:', exc)

## ۷. اجرای آزمایش‌ها
برای هر چهار مجموعه جدا اجرا کنید. پیش‌فرض ۱۰۰ تلاش تفاضلی در هر مجموعه است؛ نویز/برش، حساسیت digest، حذف مؤلفه‌ها و بازیابی دقیق هم ثبت می‌شوند. طول پیام و ظرفیت واقعی‌اند. اجرای همهٔ مجموعه‌ها ممکن است از زمان یک نشست کولب طولانی‌تر باشد.

In [ ]:
from scripts.experiments import run
from scripts.reporting import make_report
if RUN_EXPERIMENTS:
    if not METHOD_READY: raise RuntimeError('ابتدا مانع مشخصات/وزن/دینامیک را رفع و مستند کنید')
    if not PAYLOAD.is_file(): raise FileNotFoundError('فایل فراداده آزمایشی لازم است: '+str(PAYLOAD))
    status = run(MANIFEST, DATASET, CHECKPOINT, cfg, PAYLOAD, RUN_DIR, DEVICE, include_ablations=True)
    print(status)
    print(make_report(RUN_DIR))
else:
    print('اجرای پژوهشی فعال نشده است؛ نتیجه‌ای ساخته نشد.')

## ۸. رفت‌وبرگشت مستقل تصویر و پیام
فایل راز خارج از گزارش و فایل‌های ارسالی نگه‌داری می‌شود. گیرنده به stego، recovery.msr و همان راز نیاز دارد. این افزونهٔ اصلاحی با ادعای گیرندهٔ بدون فایل کمکی پایان‌نامه متفاوت است.

In [ ]:
def cli(*args):
    return subprocess.run([sys.executable, '-m', 'scripts', *map(str,args)], check=True)
RUN_DELIVERY = False
if RUN_DELIVERY:
    if not METHOD_READY: raise RuntimeError('روش آماده نیست')
    KEYFILE = WORK/'private_research.key'
    if not KEYFILE.exists(): cli('keygen','--output',KEYFILE)
    SENT = WORK/'sent_001'; RECEIVED = WORK/'received_001'
    cli('send','--manifest',MANIFEST,'--dataset',DATASET,'--id',selected['id'],'--checkpoint',CHECKPOINT,
        '--config',PROFILE,'--payload',PAYLOAD,'--keyfile',KEYFILE,'--output',SENT,'--device',DEVICE)
    cli('receive','--stego',SENT/'stego.png','--sidecar',SENT/'recovery.msr','--keyfile',KEYFILE,'--output',RECEIVED)

## ۹. برآورد لیاپانوف و پرترهٔ فاز
این برآورد زمان محدود است. طول‌های متفاوت، چند شرایط اولیه و گام‌های کوچکتر برای بررسی همگرایی لازم‌اند؛ علامت مثبت به‌تنهایی اثبات نیست.

In [ ]:
from scripts.dynamics import lyapunov
if METHOD_READY and RUN_EXPERIMENTS:
    dynamics = lyapunov(digest, cfg, steps=100000, qr_interval=10, output=WORK/'dynamics'/f'{DATASET}.json')
    print({k:v for k,v in dynamics.items() if k not in ('phase','convergence','environment')})
    if dynamics['status'] == 'ok':
        import numpy as np
        phase = np.array(dynamics['phase'])
        fig = plt.figure(); ax = fig.add_subplot(111, projection='3d')
        ax.plot(phase[:,0], phase[:,1], phase[:,2], linewidth=.4)
        ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z'); plt.show()

## ۱۰. NIST رسمی: دریافت، ساخت و اجرا
این مرحله بدون دادهٔ کافی متوقف می‌شود؛ هیچ پرکردن/تکرار تصویر برای رسیدن به صد میلیون بیت انجام نمی‌شود. خروجی تمام ۱۵ خانواده و مؤلفه‌های آن‌ها در پوشهٔ نتیجه باقی می‌ماند. دادهٔ کمتر با برچسب آزمون نرم‌افزار از این مرحلهٔ پژوهشی جدا است.

In [ ]:
os.environ['NIST_STS_URL'] = 'https://csrc.nist.gov/CSRC/media/Projects/Random-Bit-Generation/documents/sts-2_1_2.zip'
NIST_RUNS = [WORK/'runs'/f'{name}_001' for name in ['DRIVE','RITE','BraTS2020','COVID19_CXR']]
if RUN_NIST:
    import urllib.request
    from scripts.nist import build_official, export_streams, run_official
    archive = WORK/'sts-2_1_2.zip'
    if not archive.exists(): urllib.request.urlretrieve(os.environ['NIST_STS_URL'], archive)
    BUILD = WORK/'nist_build'
    if not BUILD.exists(): sts_root = build_official(archive, BUILD)
    else: sts_root = Path(json.loads((BUILD/'build.json').read_text())['assess']).parent
    nist_input = WORK/'nist_input_001'
    if not nist_input.exists(): export_streams(NIST_RUNS, nist_input, sequences=100, bits=1000000)
    nist_result = run_official(sts_root, nist_input, WORK/'nist_results_001', timeout=7200)
    print(nist_result['status'], 'families:', nist_result['families'])

## ۱۱. جمع‌بندی و دریافت جدول‌ها
فقط گزارش اجرای انتخابی بسته‌بندی می‌شود؛ فایل راز و دادهٔ خام پزشکی در ZIP گزارش نیست. جدول literature_reported.csv نقل از پایان‌نامه است و بازاجرای مقاله‌ها نیست. زمان کولب را زمان Jetson معرفی نکنید.

In [ ]:
if (RUN_DIR/'status.json').exists():
    make_report(RUN_DIR)
    import shutil
    archive = shutil.make_archive(str(WORK/f'{DATASET}_report'), 'zip', RUN_DIR/'reports')
    files.download(archive)
else:
    print('هنوز اجرای ثبت‌شده‌ای برای گزارش وجود ندارد.')